In [4]:
import os
from typing import List

from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import models, transforms

# ---------- PATH / CONFIG ----------

ROOT_DIR = "SCUT-FBP5500_v2"
IMG_DIR = os.path.join(ROOT_DIR, "Images")
ALL_LABELS_TXT = os.path.join(ROOT_DIR, "train_test_files", "All_labels.txt")

SAVE_DIR = "runs/face_score_fast"
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
NUM_EPOCHS = 10          # น้อยๆ พอ
VAL_RATIO = 0.2
SUBSET_SIZE = 800       # ใช้แค่ 800 รูป ให้ไวก่อน

# ---------- DATASET ----------

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

class SCUTFaceScoreDataset(Dataset):
    def __init__(self, img_dir: str, label_txt: str, transform=None):
        self.img_dir = img_dir
        self.transform = transform

        names: List[str] = []
        scores: List[float] = []

        with open(label_txt, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) < 2:
                    continue
                name, score_str = parts[0], parts[1]
                try:
                    score = float(score_str)
                except ValueError:
                    continue
                names.append(name)
                scores.append(score)

        self.image_names = names
        self.scores = scores

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        score = self.scores[idx]
        img_path = os.path.join(self.img_dir, img_name)

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        score_tensor = torch.tensor([score], dtype=torch.float32)
        return img, score_tensor

# ---------- MODEL ----------

class FaceScoreResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet34(
            weights=models.ResNet34_Weights.IMAGENET1K_V1
        )
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.reg_head = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        feats = self.backbone(x)
        score = self.reg_head(feats)
        return score

def main():
    full_ds = SCUTFaceScoreDataset(IMG_DIR, ALL_LABELS_TXT, transform=transform)

    # ใช้แค่ subset เล็กๆ
    subset_size = min(SUBSET_SIZE, len(full_ds))
    indices = list(range(subset_size))
    ds = Subset(full_ds, indices)

    n_total = len(ds)
    n_val = int(VAL_RATIO * n_total)
    n_train = n_total - n_val
    train_ds, val_ds = random_split(ds, [n_train, n_val])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=0)

    model = FaceScoreResNet().to(device)

    # freeze backbone ทั้งหมด train แค่ reg_head
    for p in model.backbone.parameters():
        p.requires_grad = False

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.reg_head.parameters(), lr=1e-3)

    best_val = float("inf")
    best_path = os.path.join(SAVE_DIR, "best_face_score_fast.pt")

    for epoch in range(NUM_EPOCHS):
        model.train()
        train_loss = 0.0
        for imgs, scores in train_loader:
            imgs = imgs.to(device)
            scores = scores.to(device)

            optimizer.zero_grad()
            preds = model(imgs)
            loss = criterion(preds, scores)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * imgs.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, scores in val_loader:
                imgs = imgs.to(device)
                scores = scores.to(device)
                preds = model(imgs)
                loss = criterion(preds, scores)
                val_loss += loss.item() * imgs.size(0)
        val_loss /= len(val_loader.dataset)

        print(f"Epoch {epoch+1}/{NUM_EPOCHS} "
              f"Train: {train_loss:.4f}  Val: {val_loss:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), best_path)
            print(f"  -> saved to {best_path}")

    print("Done.")

if __name__ == "__main__":
    main()


Epoch 1/10 Train: 1.5676  Val: 0.4950
  -> saved to runs/face_score_fast/best_face_score_fast.pt
Epoch 2/10 Train: 0.4641  Val: 0.3427
  -> saved to runs/face_score_fast/best_face_score_fast.pt
Epoch 3/10 Train: 0.3165  Val: 0.3198
  -> saved to runs/face_score_fast/best_face_score_fast.pt
Epoch 4/10 Train: 0.2936  Val: 0.2553
  -> saved to runs/face_score_fast/best_face_score_fast.pt
Epoch 5/10 Train: 0.2585  Val: 0.2359
  -> saved to runs/face_score_fast/best_face_score_fast.pt
Epoch 6/10 Train: 0.2539  Val: 0.2399
Epoch 7/10 Train: 0.2293  Val: 0.2245
  -> saved to runs/face_score_fast/best_face_score_fast.pt
Epoch 8/10 Train: 0.2305  Val: 0.2190
  -> saved to runs/face_score_fast/best_face_score_fast.pt
Epoch 9/10 Train: 0.2076  Val: 0.2221
Epoch 10/10 Train: 0.2098  Val: 0.3055
Done.


In [ ]:
# face_score_with_dog_webcam.py

import os
import cv2
import torch
import numpy as np
import pandas as pd
from torch import nn
from torchvision import models, transforms
from PIL import Image

# ---------- PATH CONFIG ----------

# SCUT-FBP5500 face score model (prototype)
MODEL_WEIGHTS_FACE = "runs/face_score_fast/best_face_score_fast.pt"

# dog-breed model (ของคุณเดิม)
TRAIN_DIR_DOG = "kaggle_dog_tiny/train"
LABEL_CSV_DOG = "kaggle_dog_tiny/labels.csv"
BREED_NAMES_NPY = "kaggle_dog_tiny/breed_names.npy"
MODEL_WEIGHTS_DOG = "runs/pose/train/weights/best_Dog.pt"

FACE_CASCADE_PATH = "haarcascade_frontalface_default.xml"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
devices = [device]

# ---------- CLASS NAMES (DOG) ----------

breed_names = np.load(BREED_NAMES_NPY).tolist()
class_names = breed_names          # list of dog breed strings
num_classes_dog = len(class_names)

df_dog = pd.read_csv(LABEL_CSV_DOG)   # columns: id, breed

breed_to_id = {}
for _, row in df_dog.iterrows():
    b = row["breed"]
    if b in class_names and b not in breed_to_id:
        breed_to_id[b] = row["id"]

class_idx_to_example_path = {}
for idx, breed in enumerate(class_names):
    img_id = breed_to_id.get(breed)
    if img_id is None:
        continue
    path = os.path.join(TRAIN_DIR_DOG, f"{img_id}.jpg")
    if os.path.exists(path):
        class_idx_to_example_path[idx] = path

# ---------- MODEL: FACE SCORE ----------

class FaceScoreResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet34(
            weights=models.ResNet34_Weights.IMAGENET1K_V1
        )
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        # ต้องตรงกับ train_fast: 512 -> 128 -> 1
        self.reg_head = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        feats = self.backbone(x)
        score = self.reg_head(feats)
        return score

def get_face_net(devices):
    net = FaceScoreResNet().to(devices[0])
    for p in net.backbone.parameters():
        p.requires_grad = False
    return net

# ---------- MODEL: DOG BREED (โค้ดเดิมคุณ) ----------

class FinetuneResNetDog(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = models.resnet34(
            weights=models.ResNet34_Weights.IMAGENET1K_V1
        )
        self.old_fc = self.features.fc
        self.features.fc = nn.Identity()
        self.output_new = nn.Sequential(
            nn.Linear(1000, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        feats_512 = self.features(x)
        feats_1000 = self.old_fc(feats_512)
        output = self.output_new(feats_1000)
        return output

def get_dog_net(devices, num_classes):
    net = FinetuneResNetDog(num_classes)
    net = net.to(devices[0])
    for p in net.features.parameters():
        p.requires_grad = False
    return net

# ---------- LOAD MODELS ----------

face_model = get_face_net(devices)
state_face = torch.load(MODEL_WEIGHTS_FACE, map_location=device)
face_model.load_state_dict(state_face)
face_model.eval()

dog_model = get_dog_net(devices, num_classes_dog)
state_dog = torch.load(MODEL_WEIGHTS_DOG, map_location=device)
dog_model.load_state_dict(state_dog)
dog_model.eval()

# ---------- TRANSFORM ----------

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225],
    ),
])

def preprocess_bgr_image(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    x = transform(img_pil).unsqueeze(0).to(device)
    return x

# ---------- THUMBNAILS ----------

THUMB_SIZE = (220, 220)

def load_dog_thumb_by_idx(class_idx, size=THUMB_SIZE):
    path = class_idx_to_example_path.get(class_idx)
    if path is None or not os.path.exists(path):
        return None
    img = cv2.imread(path)
    if img is None:
        return None
    thumb = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
    return thumb

def resize_face_thumb(face_roi, size=THUMB_SIZE):
    return cv2.resize(face_roi, size, interpolation=cv2.INTER_AREA)

def overlay_at(frame, thumb, x, y):
    fh, fw, _ = frame.shape
    th, tw, _ = thumb.shape

    if x >= fw or y >= fh:
        return frame

    x2 = min(x + tw, fw)
    y2 = min(y + th, fh)

    thumb_x2 = x2 - x
    thumb_y2 = y2 - y

    if thumb_x2 <= 0 or thumb_y2 <= 0:
        return frame

    frame[y:y2, x:x2] = thumb[0:thumb_y2, 0:thumb_x2]
    return frame

# ---------- FACE CASCADE ----------

face_cascade = cv2.CascadeClassifier(FACE_CASCADE_PATH)
if face_cascade.empty():
    raise RuntimeError(f"Cannot load face cascade from {FACE_CASCADE_PATH}")

# ---------- WEBCAM LOOP ----------

def run_webcam():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Cannot open webcam")

    dog_thumb_cache = {}

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(60, 60)
        )

        for (x, y, w, h) in faces:
            fh, fw, _ = frame.shape

            # ----- ขยายกรอบให้กินผม -----
            pad_x = int(w * 0.2)
            pad_y = int(h * 0.3)

            x_exp = max(x - pad_x, 0)
            y_exp = max(y - pad_y, 0)
            x2_exp = min(x + w + pad_x, fw)
            y2_exp = min(y + h , fh)

            face_roi = frame[y_exp:y2_exp, x_exp:x2_exp]

            # ----- ทำนาย face score -----
            x_face = preprocess_bgr_image(face_roi)
            with torch.no_grad():
                face_score = face_model(x_face).item()
            face_score = max(1.0, min(5.0, face_score))
            face_text = f"Face Score(MAX 5): {face_score:.1f}"

            # ----- ทำนายสายพันธุ์หมาจากหน้า (เหมือนโค้ดเดิม) -----
            x_dog = x_face  # ใช้ภาพเดียวกัน
            with torch.no_grad():
                outputs = dog_model(x_dog)
                pred_idx = outputs.argmax(dim=1).item()
            pred_label = class_names[pred_idx]

            # ----- วาดกรอบขยาย -----
            cv2.rectangle(frame, (x_exp, y_exp), (x2_exp, y2_exp),
                          (0, 255, 255), 2)

            # เขียนคะแนนและชื่อหมา
            cv2.putText(
                frame,
                face_text,
                (x_exp, max(y_exp-25, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 255),
                2,
                cv2.LINE_AA
            )
            cv2.putText(
                frame,
                pred_label,
                (x_exp, max(y_exp-5, 40)),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 255),
                2,
                cv2.LINE_AA
            )

            # ----- thumbnails -----
            face_thumb = resize_face_thumb(face_roi, THUMB_SIZE)

            if pred_idx in dog_thumb_cache:
                dog_thumb = dog_thumb_cache[pred_idx]
            else:
                dog_thumb = load_dog_thumb_by_idx(pred_idx, THUMB_SIZE)
                dog_thumb_cache[pred_idx] = dog_thumb

            base_x = x2_exp + 10
            base_y = y_exp
            if base_x + THUMB_SIZE[0]*2 + 10 > fw:
                base_x = max(x_exp - THUMB_SIZE[0]*2 - 20, 0)

            # วางรูปหน้า
            frame = overlay_at(frame, face_thumb, base_x, base_y)

            # วางรูปหมา ถัดจากหน้า
            if dog_thumb is not None:
                frame = overlay_at(
                    frame,
                    dog_thumb,
                    base_x + THUMB_SIZE[0] + 10,
                    base_y
                )

        cv2.imshow("Face Score + Dog Breed (press q to quit)", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_webcam()


: 